In [1]:
!pip install transformers torch scikit-learn -q

import pandas as pd
import torch
import os
import warnings
warnings.filterwarnings("ignore")

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from transformers import (
    AutoTokenizer, AutoModelForMaskedLM,
    VisionEncoderDecoderModel, ViTImageProcessor
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.utils import resample
from google.colab import drive

def get_linear_schedule_with_warmup(optimizer, num_warmup_steps,
                                     num_training_steps):
    def lr_lambda(s):
        if s < num_warmup_steps:
            return s / max(1, num_warmup_steps)
        return max(0.0, (num_training_steps-s) /
                   max(1, num_training_steps-num_warmup_steps))
    return LambdaLR(optimizer, lr_lambda)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print("All imported!")

Device: cuda
All imported!


In [2]:
drive.mount('/content/drive')

# Load the prompts file — already has blip_caption column
df = pd.read_csv(
    "/content/drive/MyDrive/Multimodal/multibully_with_prompts.csv"
)

# Also need image paths for ClipCap
IMG_DIR = "/content/drive/MyDrive/Multimodal/bully_data/"
df["image_path"] = df["Img-Name"].apply(
    lambda x: os.path.join(IMG_DIR, str(x))
)

print(f"Loaded: {len(df)} rows")
print(f"Images: {df['image_path'].apply(os.path.exists).sum()} found")

Mounted at /content/drive
Loaded: 5793 rows
Images: 5793 found


In [3]:
# Load ClipCap-based captioning model
# This is the same type used in PromptHate baseline

CLIPCAP_MODEL = "nlpconnect/vit-gpt2-image-captioning"

print("Loading ClipCap model...")
feature_extractor = ViTImageProcessor.from_pretrained(CLIPCAP_MODEL)
clipcap_model     = VisionEncoderDecoderModel.from_pretrained(
    CLIPCAP_MODEL
).to(device)

from transformers import AutoTokenizer as GPT2Tokenizer
gpt2_tokenizer = GPT2Tokenizer.from_pretrained(CLIPCAP_MODEL)

print(f"ClipCap loaded on: {device}")

Loading ClipCap model...


preprocessor_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.61k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  982MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

[transformers] VisionEncoderDecoderModel LOAD REPORT from: nlpconnect/vit-gpt2-image-captioning
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
decoder.transformer.h.{0...11}.attn.masked_bias           | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

ClipCap loaded on: cuda


In [4]:
# Generate ClipCap caption for one image

def generate_clipcap_caption(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        pixel_values = feature_extractor(
            images=[image], return_tensors="pt"
        ).pixel_values.to(device)

        with torch.no_grad():
            output_ids = clipcap_model.generate(
                pixel_values, max_length=30, num_beams=4
            )

        caption = gpt2_tokenizer.decode(
            output_ids[0], skip_special_tokens=True
        )
        return caption

    except Exception as e:
        return ""

# Test on one image
test_row = df.iloc[0]
blip_cap    = test_row["blip_caption"]
clipcap_cap = generate_clipcap_caption(test_row["image_path"])

print(f"Image      : {test_row['Img-Name']}")
print(f"BLIP       : {blip_cap}")
print(f"ClipCap    : {clipcap_cap}")

[transformers] We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.


Image      : 0.jpg
BLIP       : a group of women with glasses on their faces
ClipCap    : two pictures of a woman with glasses 


In [5]:
CLIPCAP_SAVE = "/content/drive/MyDrive/Multimodal/clipcap_captions.csv"
SAVE_EVERY   = 500

# Check if already partially done
if os.path.exists(CLIPCAP_SAVE):
    df_saved     = pd.read_csv(CLIPCAP_SAVE)
    already_done = df_saved[
        df_saved["clipcap_caption"].notna()
    ]["Img-Name"].tolist()
    df["clipcap_caption"] = df_saved["clipcap_caption"]
    print(f"Resuming — {len(already_done)} captions already done")
else:
    df["clipcap_caption"] = None
    already_done = []
    print("Starting fresh...")

print(f"Remaining: {len(df) - len(already_done)} images")
print("-" * 45)

import time
count  = 0
errors = 0
start  = time.time()

for idx, row in df.iterrows():
    if row["Img-Name"] in already_done:
        continue

    caption = generate_clipcap_caption(row["image_path"])
    df.at[idx, "clipcap_caption"] = caption

    if caption == "":
        errors += 1

    count += 1

    if count % SAVE_EVERY == 0:
        df.to_csv(CLIPCAP_SAVE, index=False)
        mins_left = ((time.time()-start)/count)*(len(df)-count)/60
        print(f"  {count}/{len(df)} done | "
              f"Errors: {errors} | "
              f"~{mins_left:.0f} mins left")

df.to_csv(CLIPCAP_SAVE, index=False)
print(f"\nDone! {count} captions generated")
print(f"Saved to: {CLIPCAP_SAVE}")

Starting fresh...
Remaining: 5793 images
---------------------------------------------
  500/5793 done | Errors: 0 | ~62 mins left
  1000/5793 done | Errors: 0 | ~55 mins left
  1500/5793 done | Errors: 0 | ~48 mins left
  2000/5793 done | Errors: 0 | ~42 mins left
  2500/5793 done | Errors: 0 | ~36 mins left
  3000/5793 done | Errors: 0 | ~31 mins left
  3500/5793 done | Errors: 0 | ~25 mins left
  4000/5793 done | Errors: 0 | ~20 mins left
  4500/5793 done | Errors: 0 | ~14 mins left
  5000/5793 done | Errors: 0 | ~9 mins left
  5500/5793 done | Errors: 0 | ~3 mins left

Done! 5793 captions generated
Saved to: /content/drive/MyDrive/Multimodal/clipcap_captions.csv


In [6]:
# Build Template A prompts using ClipCap captions
# Same template as BLIP experiments for fair comparison

df["prompt_clipcap"] = df.apply(
    lambda row: (
        f"The image shows {str(row['clipcap_caption'])}. "
        f"The meme text says {str(row['Img-Text'])}. "
        f"This meme is <mask>."
    ), axis=1
)

print(f"ClipCap prompts built: {len(df)}")
print(f"\nSample BLIP prompt:")
print(f"  {df['prompt_A'].iloc[0]}")
print(f"\nSample ClipCap prompt:")
print(f"  {df['prompt_clipcap'].iloc[0]}")

ClipCap prompts built: 5793

Sample BLIP prompt:
  The image shows a group of women with glasses on their faces. The meme text says Shivam @shivamishraa Girls be named naina and then have eyes that don't work. This meme is <mask>.

Sample ClipCap prompt:
  The image shows two pictures of a woman with glasses . The meme text says Shivam @shivamishraa Girls be named naina and then have eyes that don't work. This meme is <mask>.


In [7]:
SEED = 42
train_val, test_df = train_test_split(
    df, test_size=0.20,
    stratify=df["bully_label"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_val, test_size=0.125,
    stratify=train_val["bully_label"], random_state=SEED
)

bully    = train_df[train_df["bully_label"]==1]
notbully = train_df[train_df["bully_label"]==0]
train_balanced = pd.concat([
    notbully,
    resample(bully, replace=True,
             n_samples=len(notbully), random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Train (balanced): {len(train_balanced)}")
print(f"Val             : {len(val_df)}")
print(f"Test            : {len(test_df)}")

Train (balanced): 6996
Val             : 580
Test            : 1159


In [9]:
class PromptDataset(Dataset):
    def __init__(self, df, col):
        self.prompts = df[col].tolist()
        self.labels  = df["bully_label"].tolist()
    def __len__(self): return len(self.prompts)
    def __getitem__(self, idx):
        enc = tokenizer(
            self.prompts[idx], return_tensors="pt",
            truncation=True, max_length=128,
            padding="max_length"
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label"         : self.labels[idx]
        }

BATCH_SIZE = 8

# Using ClipCap prompts
train_loader = DataLoader(
    PromptDataset(train_balanced, "prompt_clipcap"),
    batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    PromptDataset(val_df, "prompt_clipcap"),
    batch_size=BATCH_SIZE, shuffle=False
)
test_loader = DataLoader(
    PromptDataset(test_df, "prompt_clipcap"),
    batch_size=BATCH_SIZE, shuffle=False
)

print(f"Train batches: {len(train_loader)}")

Train batches: 875


In [10]:
def evaluate(loader):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        with torch.no_grad():
            logits = model(
                input_ids=ids, attention_mask=attn
            ).logits
        for i in range(ids.shape[0]):
            pos = (ids[i]==tokenizer.mask_token_id)\
                  .nonzero(as_tuple=True)[0]
            if len(pos)==0:
                preds.append(0)
            else:
                ml = logits[i, pos[0], :]
                preds.append(
                    1 if ml[bully_id]>ml[normal_id] else 0
                )
        trues.extend(batch["label"].tolist())
    return f1_score(trues, preds, average="macro"), trues, preds

print("Evaluate function ready!")

Evaluate function ready!


In [12]:


MODEL_NAME = "xlm-roberta-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device)

bully_id  = tokenizer.convert_tokens_to_ids(
    tokenizer.tokenize("bully")[0]
)
normal_id = tokenizer.convert_tokens_to_ids(
    tokenizer.tokenize("normal")[0]
)

print(f"Model reloaded on: {device}")
print(f"bully  token id: {bully_id}")
print(f"normal token id: {normal_id}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model reloaded on: cuda
bully  token id: 50208
normal token id: 3638


In [13]:
NUM_EPOCHS = 5
optimizer  = AdamW(
    model.parameters(), lr=1e-5, weight_decay=0.01
)
scheduler  = get_linear_schedule_with_warmup(
    optimizer, 100, (len(train_loader)//2)*NUM_EPOCHS
)

best_f1, best_state = 0, None
scaler = torch.amp.GradScaler("cuda") if device=="cuda" else None

print("Training with ClipCap prompts...")
print("-" * 50)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for i, batch in enumerate(train_loader):
        ids    = batch["input_ids"].to(device)
        attn   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        target = torch.where(
            labels==1,
            torch.tensor(bully_id,  device=device),
            torch.tensor(normal_id, device=device)
        )
        lbl = torch.full(ids.shape, -100, device=device)
        lbl[(ids==tokenizer.mask_token_id)] = \
            target.repeat_interleave(
                (ids==tokenizer.mask_token_id).sum(dim=1)
            )

        if scaler:
            with torch.amp.autocast("cuda"):
                loss = model(
                    input_ids=ids, attention_mask=attn,
                    labels=lbl
                ).loss / 2
            scaler.scale(loss).backward()
        else:
            loss = model(
                input_ids=ids, attention_mask=attn,
                labels=lbl
            ).loss / 2
            loss.backward()

        total_loss += loss.item() * 2

        if (i+1) % 2 == 0:
            if scaler:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 1.0
                )
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 1.0
                )
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        del ids, attn, labels
        if device=="cuda": torch.cuda.empty_cache()

    val_f1, _, _ = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Loss: {total_loss/len(train_loader):.4f} | "
          f"Val F1: {val_f1*100:.2f}%")

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_state = {
            k: v.clone() for k, v in model.state_dict().items()
        }
        print(f"  ★ Best saved!")

model.load_state_dict(best_state)
print(f"\nDone! Best Val F1: {best_f1*100:.2f}%")

Training with ClipCap prompts...
--------------------------------------------------
Epoch 1/5 | Loss: 1.1852 | Val F1: 47.85%
  ★ Best saved!
Epoch 2/5 | Loss: 0.4905 | Val F1: 56.13%
  ★ Best saved!
Epoch 3/5 | Loss: 0.2482 | Val F1: 58.67%
  ★ Best saved!
Epoch 4/5 | Loss: 0.1458 | Val F1: 56.60%
Epoch 5/5 | Loss: 0.0917 | Val F1: 57.23%

Done! Best Val F1: 58.67%


In [14]:
test_f1, y_true, y_pred = evaluate(test_loader)

print("=== ABLATION STUDY 2 — BLIP vs ClipCap ===")
print(f"\nBLIP captions    : F1 = 61.74%")
print(f"ClipCap captions : F1 = {test_f1*100:.2f}%")
print(f"Difference       : {(61.74 - test_f1*100):+.2f}%")
print()
print(classification_report(
    y_true, y_pred,
    target_names=["Not-Bully", "Bully"]
))

bully_caught = sum(
    t==1 and p==1 for t,p in zip(y_true, y_pred)
)
print(f"Bully caught: {bully_caught} / "
      f"{sum(t==1 for t in y_true)}")

# Save predictions
test_df_save = test_df.copy()
test_df_save["pred_clipcap"] = y_pred
test_df_save.to_csv(
    "/content/drive/MyDrive/Multimodal/predictions_clipcap.csv",
    index=False
)
print("\nPredictions saved to Drive!")

=== ABLATION STUDY 2 — BLIP vs ClipCap ===

BLIP captions    : F1 = 61.74%
ClipCap captions : F1 = 59.97%
Difference       : +1.77%

              precision    recall  f1-score   support

   Not-Bully       0.89      0.89      0.89      1000
       Bully       0.31      0.31      0.31       159

    accuracy                           0.81      1159
   macro avg       0.60      0.60      0.60      1159
weighted avg       0.81      0.81      0.81      1159

Bully caught: 50 / 159

Predictions saved to Drive!
